# Connect Garmin to Cadence
1. In Cadence, open **Profile → Garmin → Connect** and keep the pairing code on screen.
2. Here, tap **▶** on the cell below (Colab may ask you to sign in with Google and to confirm running the notebook).
3. Answer the questions that appear under the cell. Your Garmin password goes only to Garmin, never to Cadence.


In [ ]:
#@title Connect Garmin to Cadence — press ▶ and answer the questions below
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'garminconnect', 'curl_cffi'], check=True)

import getpass, json, requests
from garminconnect import Garmin

CADENCE = 'https://cadence-xi-one.vercel.app'
pair = input('Pairing code shown in Cadence (like K7Q2-M9XD): ').strip()
email = input('Garmin email: ').strip()
password = getpass.getpass('Garmin password: ')
g = Garmin(email, password, prompt_mfa=lambda: input('Code Garmin just sent you: ').strip())
print('Signing in to Garmin… (this can take up to a minute)')
g.login()
password = None
print('Signed in. Sending the connection to Cadence…')
r = requests.post(CADENCE + '/api/report', json={'action': 'garmin-pair', 'code': pair, 'tokens': g.client.dumps()}, timeout=90)
res = r.json() if r.headers.get('content-type', '').startswith('application/json') else {'error': r.text[:200]}
if res.get('ok'):
    print(f"✅ Done. Cadence brought in {res.get('activities', 0)} activities and {res.get('days', 0)} days of sleep data. Go back to the app.")
else:
    print('❌ ' + str(res.get('error') or r.status_code))
